# Obligatorio - Taller de Deep Learning
---

- Diciembre 2025  
- Enrique Oliva - 214205

## Introducción
---

Este notebook implementa la arquitectura U-Net desde cero en PyTorch para segmentación binaria de personas, desarrollado como obligatorio del Taller de Deep Learning. Basado en el paper de Ronneberger et al. (2015), el modelo utiliza un encoder-decoder de 4 niveles con skip connections, InstanceNorm para estabilidad en batches pequeños, y Dropout en el bottleneck como regularización. Las imágenes originales de 800×800 se procesan a 384×384 durante entrenamiento, con augmentación adaptada al dataset (transformaciones geométricas y de color realistas). El entrenamiento emplea BCEDiceLoss, scheduler de learning rate con decaimiento coseno, y un sistema de checkpoints que preserva el mejor modelo. La inferencia incluye Test-Time Augmentation y genera predicciones en formato RLE para la competencia de Kaggle, donde se requiere un coeficiente Dice mínimo de 0.75.

Paper:
- https://arxiv.org/abs/1505.04597

## Configuración general
---

### Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import wandb

### Configuración del dispositivo (GPU/CPU) y seeds para reproducibilidad. Verificamos si tenemos CUDA disponible para entrenar en GPU

In [ ]:
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("No se detectó GPU")

### Creamos la carpeta results/ con subdirectorios para organizar outputs

In [ ]:
BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data' / 'tdl-obligatorio-2025'
TRAIN_IMG_DIR = DATA_DIR / 'train' / 'images'
TRAIN_MASK_DIR = DATA_DIR / 'train' / 'masks'
TEST_IMG_DIR = DATA_DIR / 'test' / 'images'

RESULTS_DIR = BASE_DIR / 'results'

RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / 'bestModel').mkdir(exist_ok=True)
(RESULTS_DIR / 'graphs').mkdir(exist_ok=True)
(RESULTS_DIR / 'predictions').mkdir(exist_ok=True)
(RESULTS_DIR / 'kaggleSubmission').mkdir(exist_ok=True)

print("Directorio de resultados creado:")
print(f"  - {RESULTS_DIR}/")
print(f"    ├── bestModel/")
print(f"    ├── graphs/")
print(f"    ├── predictions/")
print(f"    └── kaggleSubmission/")
print(f"\nDataset ubicado en: {DATA_DIR}")
print(f"  - Train images: {len(list(TRAIN_IMG_DIR.glob('*.png')))}")
print(f"  - Train masks: {len(list(TRAIN_MASK_DIR.glob('*.png')))}")
print(f"  - Test images: {len(list(TEST_IMG_DIR.glob('*.png')))}")

### Hiperparámetros

In [ ]:
IMG_SIZE = 384
ORIG_SIZE = 800
BATCH_SIZE = 8
NUM_WORKERS = 0
VAL_SPLIT = 0.2

LEARNING_RATE = 1e-4
NUM_EPOCHS = 700

UNET_DEPTH = 4
UNET_FILTERS = [64, 128, 256, 512, 1024]

THRESHOLD_RANGE = np.arange(0.3, 0.8, 0.05)

USE_WANDB = True
WANDB_PROJECT = "Obligatorio de Taller de IA 2025"
WANDB_GROUP = datetime.now().strftime("%d\%m\%y - %H:%M")

CONFIG = {
    'img_size': IMG_SIZE,
    'orig_size': ORIG_SIZE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'val_split': VAL_SPLIT,
    'learning_rate': LEARNING_RATE,
    'num_epochs': NUM_EPOCHS,
    'unet_depth': UNET_DEPTH,
    'unet_filters': UNET_FILTERS,
}

## Análisis del dataset
---

### Funciones auxiliares para visualizar imágenes con máscaras

In [ ]:
def visualize_sample(image, mask, title="", save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Imagen Original')
    axes[0].axis('off')
    
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Máscara')
    axes[1].axis('off')
    
    axes[2].imshow(image)
    axes[2].imshow(mask, alpha=0.5, cmap='Reds')
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    if title:
        fig.suptitle(title, fontsize=16)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=100)
        plt.show()
    else:
        plt.show()

def load_image_and_mask(image_path, mask_path):
    image = np.array(Image.open(image_path).convert('RGB'))
    mask = np.array(Image.open(mask_path).convert('L'))
    return image, mask

print("Funciones auxiliares definidas")

### Obtenemos todos los IDs de las imágenes de entrenamiento

In [ ]:
train_image_paths = sorted(list(TRAIN_IMG_DIR.glob('*.png')))
train_mask_paths = sorted(list(TRAIN_MASK_DIR.glob('*.png')))
test_image_paths = sorted(list(TEST_IMG_DIR.glob('*.png')))

train_ids = [p.stem for p in train_image_paths]
test_ids = [p.stem for p in test_image_paths]

print(f"Total imágenes entrenamiento: {len(train_ids)}")
print(f"Total imágenes test: {len(test_ids)}")
print(f"\nPrimeros IDs de entrenamiento: {train_ids[:5]}")
print(f"Últimos IDs de entrenamiento: {train_ids[-5:]}")

### Verificamos las dimensiones de las imágenes. Según el obligatorio, todas deberían ser 800x800

In [ ]:
sample_img_path = train_image_paths[0]
sample_mask_path = train_mask_paths[0]

sample_img = Image.open(sample_img_path)
sample_mask = Image.open(sample_mask_path)

print(f"Dimensiones imagen: {sample_img.size}")
print(f"Dimensiones máscara: {sample_mask.size}")
print(f"Modo imagen: {sample_img.mode}")
print(f"Modo máscara: {sample_mask.mode}")

img_array = np.array(sample_img)
mask_array = np.array(sample_mask)

print(f"\nShape imagen array: {img_array.shape}")
print(f"Shape máscara array: {mask_array.shape}")
print(f"Valores únicos en máscara: {np.unique(mask_array)}")
print(f"Rango valores imagen: [{img_array.min()}, {img_array.max()}]")

### Visualizamos algunas muestras del dataset para ver cómo se ven

In [ ]:
np.random.seed(SEED)
sample_indices = np.random.choice(len(train_ids), 6, replace=False)

for i, idx in enumerate(sample_indices):
    img_path = train_image_paths[idx]
    mask_path = train_mask_paths[idx]
    
    image, mask = load_image_and_mask(img_path, mask_path)
    
    visualize_sample(image, mask, 
                    title=f'Muestra {i+1} (ID: {train_ids[idx]})')

### Estadísticas básicas del dataset. Verificamos que todas las imágenes sean del mismo tamaño

In [ ]:
all_shapes = []
for img_path in train_image_paths[:100]:
    img = Image.open(img_path)
    all_shapes.append(img.size)

unique_shapes = set(all_shapes)

print(f"Dimensiones únicas en muestra de 100 imágenes: {unique_shapes}")

if len(unique_shapes) == 1:
    print("Todas las imágenes tienen el mismo tamaño")
    img_width, img_height = unique_shapes.pop()
    print(f"  Tamaño: {img_width}x{img_height}")
else:
    print("Advertencia: Las imágenes tienen diferentes tamaños")

dataset_stats = {
    'num_train_images': len(train_ids),
    'num_test_images': len(test_ids),
    'image_size': [img_width, img_height],
    'seed': SEED
}

print(f"\nEstadísticas del dataset:")
for key, value in dataset_stats.items():
    print(f"  {key}: {value}")

### Análisis de balance de clases: cuántos píxeles son persona vs fondo

In [ ]:
print("Análisis de balance de clases en 200 muestras aleatorias")

np.random.seed(SEED)
sample_mask_indices = np.random.choice(len(train_mask_paths), 200, replace=False)

foreground_ratios = []

for idx in sample_mask_indices:
    mask = np.array(Image.open(train_mask_paths[idx]).convert('L'))
    mask_binary = (mask > 0).astype(np.uint8)
    
    total_pixels = mask_binary.size
    foreground_pixels = mask_binary.sum()
    foreground_ratio = foreground_pixels / total_pixels
    
    foreground_ratios.append(foreground_ratio)

foreground_ratios = np.array(foreground_ratios)

print(f"\nEstadísticas de balance de clases:")
print(f"  tasa foreground promedio: {foreground_ratios.mean():.4f}")
print(f"  tasa foreground mediano: {np.median(foreground_ratios):.4f}")
print(f"  tasa foreground min: {foreground_ratios.min():.4f}")
print(f"  tasa foreground max: {foreground_ratios.max():.4f}")
print(f"  tasa foreground std: {foreground_ratios.std():.4f}")

dataset_stats['foreground_ratio_mean'] = float(foreground_ratios.mean())
dataset_stats['foreground_ratio_std'] = float(foreground_ratios.std())

if foreground_ratios.mean() < 0.3:
    print("\nDataset desbalanceado")
else:
    print("\nDataset relativamente balanceado")

### Análisis de distribución de tamaños de objetos. Miramos cuánto espacio ocupan las personas en cada imagen

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(foreground_ratios, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Ratio de píxeles foreground')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de tamaño de objetos en el dataset')
ax.axvline(foreground_ratios.mean(), color='red', linestyle='--', 
           label=f'Media: {foreground_ratios.mean():.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"La mayoría de imágenes tienen ~{foreground_ratios.mean()*100:.1f}% de píxeles de persona")

### Visualizamos casos extremos: imágenes con muy poco y mucho foreground, para entender mejor la variabilidad del dataset

In [ ]:
min_idx = sample_mask_indices[np.argmin(foreground_ratios)]
max_idx = sample_mask_indices[np.argmax(foreground_ratios)]

print(f"Caso con mínimo foreground: {foreground_ratios.min()*100:.2f}%")
img_min, mask_min = load_image_and_mask(train_image_paths[min_idx], 
                                         train_mask_paths[min_idx])
visualize_sample(img_min, mask_min, 
                title=f'Mínimo Foreground ({foreground_ratios.min()*100:.2f}%)')

print(f"Caso con máximo foreground: {foreground_ratios.max()*100:.2f}%")
img_max, mask_max = load_image_and_mask(train_image_paths[max_idx], 
                                         train_mask_paths[max_idx])
visualize_sample(img_max, mask_max, 
                title=f'Máximo Foreground ({foreground_ratios.max()*100:.2f}%)')

### Resumen del análisis del dataset

In [ ]:
print(f"Resumen del análisis:")
print(f"  - {dataset_stats['num_train_images']} imágenes de entrenamiento")
print(f"  - {dataset_stats['num_test_images']} imágenes de test")
print(f"  - Tamaño: {dataset_stats['image_size']}")
print(f"  - Foreground ratio: {dataset_stats['foreground_ratio_mean']:.4f} ± {dataset_stats['foreground_ratio_std']:.4f}")

## Procesamiento del dataset
---

### Data augmentation con Albumentations

In [ ]:
_grayscale_flag = [False]

def detect_grayscale(image, **kwargs):
    r, g, b = image[:,:,0], image[:,:,1], image[:,:,2]
    _grayscale_flag[0] = np.allclose(r, g, atol=5) and np.allclose(g, b, atol=5)
    return image

def restore_grayscale(image, **kwargs):
    if _grayscale_flag[0]:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    return image

train_transform = A.Compose([
    A.RandomResizedCrop(
        size=(IMG_SIZE, IMG_SIZE),
        scale=(0.5, 1.0),
        ratio=(1.0, 1.0),
        p=1.0
    ),

    A.Lambda(image=detect_grayscale),

    A.HorizontalFlip(p=0.5),

    A.OneOf([
        A.RandomBrightnessContrast(
            brightness_limit=0.15,
            contrast_limit=0.15,
            p=1.0
        ),
        A.RandomGamma(
            gamma_limit=(85, 115),
            p=1.0
        ),
    ], p=0.5),

    A.OneOf([
        A.HueSaturationValue(
            hue_shift_limit=8,
            sat_shift_limit=15,
            val_shift_limit=12,
            p=1.0
        ),
        A.ColorJitter(
            brightness=0.08,
            contrast=0.08,
            saturation=0.12,
            hue=0.05,
            p=1.0
        ),
    ], p=0.35),

    A.ToGray(p=0.20),

    A.Lambda(image=restore_grayscale),

    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print("Data augmentation - Transformaciones:")
print("  RandomResizedCrop")
print("  Deteccion y restauracion de escala de grises")
print("  HorizontalFlip")
print("  OneOf[RandomBrightnessContrast/RandomGamma]")
print("  OneOf[HueSaturationValue/ColorJitter]")
print("  ToGray")

### Testeamos el augmentation en una imagen

In [ ]:
test_idx = 0
test_img, test_mask = load_image_and_mask(train_image_paths[test_idx], 
                                          train_mask_paths[test_idx])

print(f"Imagen original: {test_img.shape}, Máscara original: {test_mask.shape}")

test_mask_binary = (test_mask > 0).astype(np.uint8)

transformed = train_transform(image=test_img, mask=test_mask_binary)
aug_img = transformed['image']
aug_mask = transformed['mask']

print(f"Imagen transformada: {aug_img.shape}, Máscara transformada: {aug_mask.shape}")
print(f"Rango imagen (tensor): [{aug_img.min():.3f}, {aug_img.max():.3f}]")
print(f"Valores únicos máscara: {torch.unique(aug_mask)}")

### Generamos varios ejemplos de augmentation para visualizar

In [ ]:
sample_img, sample_mask = load_image_and_mask(train_image_paths[1334], 
                                               train_mask_paths[1334])
sample_mask = (sample_mask > 0).astype(np.uint8)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

for i in range(11):
    row = (i + 1) // 4
    col = (i + 1) % 4
    
    augmented = train_transform(image=sample_img, mask=sample_mask)
    aug_img = augmented['image']
    aug_mask = augmented['mask']
    
    aug_img_np = aug_img.permute(1, 2, 0).numpy()
    aug_img_np = aug_img_np * IMAGENET_STD + IMAGENET_MEAN
    aug_img_np = np.clip(aug_img_np, 0, 1)
    
    axes[row, col].imshow(aug_img_np)
    axes[row, col].set_title(f'Aug {i+1}')
    axes[row, col].axis('off')

plt.suptitle('Ejemplos de Data Augmentation', fontsize=16)
plt.tight_layout()
plt.show()

### Verificación: las máscaras mantienen valores binarios 0/1. Si aparecen valores intermedios, hay un error en la interpolación

In [ ]:
num_checks = 20
all_correct = True

for i in range(num_checks):
    idx = np.random.randint(len(train_image_paths))
    img, mask = load_image_and_mask(train_image_paths[idx], train_mask_paths[idx])
    mask = (mask > 0).astype(np.uint8)
    
    augmented = train_transform(image=img, mask=mask)
    aug_mask = augmented['mask']
    
    unique_vals = torch.unique(aug_mask)
    
    if len(unique_vals) > 2:
        print(f"ERROR en muestra {i}: valores {unique_vals}")
        all_correct = False

if all_correct:
    print(f"{num_checks} verificaciones pasadas")
    print("Las máscaras mantienen valores binarios correctamente")
else:
    print("Advertencia: Revisar configuración de augmentation")

### Clase Dataset para cargar imágenes y máscaras. Soporta train/val/test modes

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.is_test = (mask_paths is None)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert('RGB'))
        
        if self.is_test:
            if self.transform:
                transformed = self.transform(image=image)
                image = transformed['image']
            return image
        else:
            mask = np.array(Image.open(self.mask_paths[idx]).convert('L'))
            mask = (mask > 0).astype(np.uint8)
            
            if self.transform:
                transformed = self.transform(image=image, mask=mask)
                image = transformed['image']
                mask = transformed['mask']
            else:
                image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
                mask = torch.from_numpy(mask).float()
            
            mask = mask.unsqueeze(0) if mask.dim() == 2 else mask
            
            return image, mask

### Creamos un dataset de prueba para verificar que funciona

In [ ]:
test_dataset = SegmentationDataset(
    image_paths=train_image_paths[:10],
    mask_paths=train_mask_paths[:10],
    transform=train_transform
)

print(f"Dataset de prueba creado: {len(test_dataset)} muestras")

sample_img, sample_mask = test_dataset[0]

print(f"\nPrimera muestra:")
print(f"  Imagen shape: {sample_img.shape}")
print(f"  Máscara shape: {sample_mask.shape}")
print(f"  Imagen dtype: {sample_img.dtype}")
print(f"  Máscara dtype: {sample_mask.dtype}")
print(f"  Imagen range: [{sample_img.min():.3f}, {sample_img.max():.3f}]")
print(f"  Máscara unique: {torch.unique(sample_mask)}")

### Testeamos el DataLoader con un batch

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

batch_images, batch_masks = next(iter(test_loader))

print(f"Batch cargado:")
print(f"  Imágenes shape: {batch_images.shape}")
print(f"  Máscaras shape: {batch_masks.shape}")
print(f"  Imágenes dtype: {batch_images.dtype}")
print(f"  Máscaras dtype: {batch_masks.dtype}")

expected_img_shape = (4, 3, IMG_SIZE, IMG_SIZE)
expected_mask_shape = (4, 1, IMG_SIZE, IMG_SIZE)

if batch_images.shape == expected_img_shape and batch_masks.shape == expected_mask_shape:
    print(f"\nShapes correctos")
    print(f"  Esperado imágenes: {expected_img_shape} ")
    print(f"  Esperado máscaras: {expected_mask_shape} ")
else:
    print(f"\nAdvertencia: Shapes inesperados")
    print(f"  Esperado imágenes: {expected_img_shape}, Obtenido: {batch_images.shape}")
    print(f"  Esperado máscaras: {expected_mask_shape}, Obtenido: {batch_masks.shape}")

### Visualizamos un batch para verificar que todo se ve bien

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    img = batch_images[i].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    
    mask = batch_masks[i, 0].numpy()
    
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Imagen {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(img)
    axes[1, i].imshow(mask, alpha=0.5, cmap='Reds')
    axes[1, i].set_title(f'Con Máscara {i+1}')
    axes[1, i].axis('off')

plt.suptitle('Batch de Entrenamiento', fontsize=16)
plt.tight_layout()
plt.show()

## Implementación del modelo
---

### Doble convolución
- Conv 3x3 -> InstanceNorm -> ReLU (x2)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)

  ### Bloque encoder
  - MaxPool 2x2 + doble convolución

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Encoder, self).__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)

### Bloque decoder
- Upsample + concatenar skip + doble convolución

In [ ]:
class Decoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Decoder, self).__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv_reduce = nn.Conv2d(in_channels, in_channels // 2, kernel_size=1)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.conv_reduce(x1)
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

  ### Arquitectura U-Net completa  
  - Encoder: 64→128→256→512→1024  
  - Decoder: 1024→512→256→128→64→1 (con skip connections)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, dropout_p=0.1):
        super(UNet, self).__init__()
        
        self.inc = DoubleConv(in_channels, 64)
        self.encoder1 = Encoder(64, 128)
        self.encoder2 = Encoder(128, 256)
        self.encoder3 = Encoder(256, 512)
        self.encoder4 = Encoder(512, 1024)
        
        self.dropout = nn.Dropout2d(p=dropout_p)
        
        self.decoder1 = Decoder(1024, 512)
        self.decoder2 = Decoder(512, 256)
        self.decoder3 = Decoder(256, 128)
        self.decoder4 = Decoder(128, 64)
        
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.encoder1(x1)
        x3 = self.encoder2(x2)
        x4 = self.encoder3(x3)
        x5 = self.encoder4(x4)
        
        x5 = self.dropout(x5)
        
        x = self.decoder1(x5, x4)
        x = self.decoder2(x, x3)
        x = self.decoder3(x, x2)
        x = self.decoder4(x, x1)
        
        logits = self.outc(x)
        return logits

### Instanciamos el modelo y lo movemos a GPU

In [ ]:
model = UNet(in_channels=3, out_channels=1)
model = model.to(device)

print(f"Modelo creado y movido a {device}")

### Resumen de la arquitectura del modelo

In [ ]:
def get_model_summary(model, input_size):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Resumen del modelo:")
    print(f"  Total parámetros: {total_params:,}")
    print(f"  Parámetros entrenables: {trainable_params:,}")
    print(f"  Tamaño de entrada: {input_size}")
    
    return total_params, trainable_params

total_params, trainable_params = get_model_summary(model, (BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE))

model_info = {
    'total_params': total_params,
    'trainable_params': trainable_params,
    'architecture': 'UNet',
    'depth': 4,
    'filters': [64, 128, 256, 512, 1024]
}

### Test forward pass con un tensor dummy. Verificamos que el modelo puede procesar un batch

In [ ]:
dummy_input = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE).to(device)

print(f"Input shape: {dummy_input.shape}")

with torch.no_grad():
    output = model(dummy_input)

print(f"Output shape: {output.shape}")

expected_output_shape = (BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE)

if output.shape == expected_output_shape:
    print(f"\nForward pass correcto")
    print(f"Esperado: {expected_output_shape}")
    print(f"Obtenido: {output.shape}")
else:
    print(f"\nAdvertencia: Output shape inesperado")
    print(f"Esperado: {expected_output_shape}")
    print(f"Obtenido: {output.shape}")

del dummy_input, output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### Análisis detallado de parámetros por capa

In [ ]:
def count_parameters_by_layer(model):
    print("Parámetros por módulo:")
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            num_params = param.numel()
            total += num_params
            if num_params > 100000:
                print(f"  {name:50s}: {num_params:>12,}")
    
    print(f"\n{'Total':50s}: {total:>12,}")
    return total

total_counted = count_parameters_by_layer(model)

model_info['params_breakdown'] = total_counted

### Dice Loss: función de pérdida basada en el coeficiente de Dice  
- Es buena para datasets desbalanceados porque normaliza por el tamaño del objeto  
- Formula: 1 - (2 * intersection + smooth) / (sum + smooth)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        predictions = torch.sigmoid(predictions)
        
        predictions = predictions.view(-1)
        targets = targets.float().view(-1)
        
        intersection = (predictions * targets).sum()
        dice = (2. * intersection + self.smooth) / (predictions.sum() + targets.sum() + self.smooth)
        
        return 1 - dice

### BCE + Dice Loss: combinación de ambas pérdidas
- BCE da gradientes suaves, Dice maneja el desbalance

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super(BCEDiceLoss, self).__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    
    def forward(self, predictions, targets):
        targets = targets.float()
        
        bce_loss = self.bce(predictions, targets)
        dice_loss = self.dice(predictions, targets)
        
        combined_loss = self.bce_weight * bce_loss + self.dice_weight * dice_loss
        
        return combined_loss

### Dice Coefficient como métrica (no como loss)


In [ ]:
def dice_coefficient(predictions, targets, threshold=0.5, smooth=1.0):
    predictions = torch.sigmoid(predictions)
    predictions = (predictions > threshold).float()
    
    predictions = predictions.view(-1)
    targets = targets.float().view(-1)
    
    intersection = (predictions * targets).sum()
    dice = (2. * intersection + smooth) / (predictions.sum() + targets.sum() + smooth)
    
    return dice.item()

def batch_dice_coefficient(predictions, targets, threshold=0.5):
    batch_size = predictions.shape[0]
    dice_scores = []
    
    for i in range(batch_size):
        dice = dice_coefficient(predictions[i:i+1], targets[i:i+1], threshold)
        dice_scores.append(dice)
    
    return np.mean(dice_scores)

### Testeamos las funciones de loss con tensors dummy

In [ ]:
test_preds = torch.randn(4, 1, IMG_SIZE, IMG_SIZE)
test_targets = torch.randint(0, 2, (4, 1, IMG_SIZE, IMG_SIZE)).float()

dice_loss_fn = DiceLoss()
bce_dice_loss_fn = BCEDiceLoss()

dice_loss_value = dice_loss_fn(test_preds, test_targets)
bce_dice_loss_value = bce_dice_loss_fn(test_preds, test_targets)
dice_coef = batch_dice_coefficient(test_preds, test_targets)

print("Test de funciones de loss:")
print(f"  Dice Loss: {dice_loss_value.item():.4f}")
print(f"  BCE+Dice Loss: {bce_dice_loss_value.item():.4f}")
print(f"  Dice Coefficient: {dice_coef:.4f}")

if 0 <= dice_loss_value.item() <= 1:
    print("\nDice Loss en rango válido [0, 1]")
if 0 <= dice_coef <= 1:
    print("Dice Coefficient en rango válido [0, 1]")

del test_preds, test_targets

### Función para entrenar una época

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    model.train()
    
    running_loss = 0.0
    running_dice = 0.0
    num_batches = len(dataloader)
    
    for batch_idx, (images, masks) in enumerate(dataloader):
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item()
        
        with torch.no_grad():
            dice = batch_dice_coefficient(outputs, masks)
            running_dice += dice
        
    
    epoch_loss = running_loss / num_batches
    epoch_dice = running_dice / num_batches
    
    
    return epoch_loss, epoch_dice

### Función para validar una época

In [ ]:
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    
    running_loss = 0.0
    running_dice = 0.0
    num_batches = len(dataloader)
    
    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(dataloader):
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            running_loss += loss.item()
            dice = batch_dice_coefficient(outputs, masks)
            running_dice += dice
            
    
    epoch_loss = running_loss / num_batches
    epoch_dice = running_dice / num_batches
    
    
    return epoch_loss, epoch_dice

### Función para guardar checkpoints del modelo

In [ ]:
def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, train_dice, val_dice, filepath):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_dice': train_dice,
        'val_dice': val_dice
    }
    torch.save(checkpoint, filepath)

def load_checkpoint(model, optimizer, filepath, device):
    checkpoint = torch.load(filepath, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    return checkpoint['epoch'], checkpoint['train_loss'], checkpoint['val_loss']

### Función para graficar las curvas de entrenamiento

In [ ]:
def plot_training_curves(history, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss durante Entrenamiento')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['train_dice'], 'b-', label='Train Dice', linewidth=2)
    ax2.plot(epochs, history['val_dice'], 'r-', label='Val Dice', linewidth=2)
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Dice Coefficient')
    ax2.set_title('Dice Coefficient durante Entrenamiento')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=100)
    plt.show()

### Split del dataset de train en train y validación
- Usamos 80% para entrenar y 20% para validar

In [ ]:
num_train = len(train_image_paths)
num_val = int(num_train * VAL_SPLIT)
num_train_final = num_train - num_val

indices = list(range(num_train))
np.random.seed(SEED)
np.random.shuffle(indices)

train_indices = indices[num_val:]
val_indices = indices[:num_val]

train_img_paths = [train_image_paths[i] for i in train_indices]
train_msk_paths = [train_mask_paths[i] for i in train_indices]

val_img_paths = [train_image_paths[i] for i in val_indices]
val_msk_paths = [train_mask_paths[i] for i in val_indices]

print(f"Split del dataset:")
print(f"  Total: {num_train} imágenes")
print(f"  Train: {len(train_img_paths)} ({(1-VAL_SPLIT)*100:.0f}%)")
print(f"  Val: {len(val_img_paths)} ({VAL_SPLIT*100:.0f}%)")

split_info = {
    'total': num_train,
    'train': len(train_img_paths),
    'val': len(val_img_paths),
    'val_split': VAL_SPLIT
}

### Creamos los DataLoaders finales para train y val

In [ ]:
train_dataset = SegmentationDataset(
    image_paths=train_img_paths,
    mask_paths=train_msk_paths,
    transform=train_transform
)

val_dataset = SegmentationDataset(
    image_paths=val_img_paths,
    mask_paths=val_msk_paths,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

print("DataLoaders creados:")
print(f"  Train: {len(train_loader)} batches de {BATCH_SIZE}")
print(f"  Val: {len(val_loader)} batches de {BATCH_SIZE}")

### Verificamos que los tamaños son correctos

In [ ]:
print("Verificación de tamaños:")
print(f"  Dataset train: {len(train_dataset)} imágenes")
print(f"  Dataset val: {len(val_dataset)} imágenes")
print(f"  Total: {len(train_dataset) + len(val_dataset)}")

expected_total = len(train_image_paths)
actual_total = len(train_dataset) + len(val_dataset)

if actual_total == expected_total:
    print(f"\nTotal correcto: {actual_total} = {expected_total}")
else:
    print(f"\nAdvertencia: Total {actual_total} != {expected_total}")

print(f"\nIteraciones por época:")
print(f"  Train: {len(train_loader)} iteraciones")
print(f"  Val: {len(val_loader)} iteraciones")
print(f"  Total por época: {len(train_loader) + len(val_loader)} iteraciones")

### Visualizamos un batch final de train para confirmar que no hay problemas

In [ ]:
final_batch_images, final_batch_masks = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(min(4, BATCH_SIZE)):
    img = final_batch_images[i].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    
    mask = final_batch_masks[i, 0].numpy()
    
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Train Imagen {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(img)
    axes[1, i].imshow(mask, alpha=0.5, cmap='Reds')
    axes[1, i].set_title(f'Train + Máscara {i+1}')
    axes[1, i].axis('off')

plt.suptitle('Batch Final de Entrenamiento', fontsize=16)
plt.tight_layout()
plt.show()

print(f"Batch shape:")
print(f"  Imágenes: {final_batch_images.shape}")
print(f"  Máscaras: {final_batch_masks.shape}")

### Creamos el optimizador y la función de pérdida

In [ ]:
criterion = BCEDiceLoss(bce_weight=0.3, dice_weight=0.7)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Optimizador: Adam")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Parámetros a optimizar: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print(f"\nFunción de pérdida: BCE + Dice")
print(f"  BCE weight: 0.3")
print(f"  Dice weight: 0.7")

### Creamos el scheduler con Cosine Annealing (decaimiento suave del LR siguiendo una curva coseno)

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=5e-7
)

print(f"Scheduler: CosineAnnealingLR")
print(f"  T_max: {NUM_EPOCHS} épocas (una curva completa)")
print(f"  eta_min: 5e-7")

print("\nScheduler listo")

## Entrenamiento del modelo
---

### Setup de Mixed Precision Training para acelerar el entrenamiento
Mixed Precision Training (AMP - Automatic Mixed Precision):
- Usa float16 para cálculos forward/backward (más rápido, menos memoria)
- Mantiene float32 para actualizaciones de pesos (precisión)
- GradScaler evita underflow en gradientes float16

### Loop de entrenamiento

Usamos Mixed Precision Training para acelerar el entrenamiento

- Usa float16 para cálculos forward/backward (más rápido, menos memoria)
- Mantiene float32 para actualizaciones de pesos (precisión)
- GradScaler evita underflow en gradientes float16

In [ ]:
from IPython.display import clear_output
from datetime import datetime

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda') if use_amp else None

if USE_WANDB:
    if wandb.run is not None:
        wandb.finish()
    wandb.login()
    wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=WANDB_GROUP,
        config=CONFIG
    )

print(f"Épocas: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Device: {device}")
print(f"wandb: {'Activado' if USE_WANDB else 'Desactivado'}")

history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'lr': []
}

best_val_dice = 0.0

def save_training_graphs(history, graphs_dir):
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.set_title('Loss durante Entrenamiento')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graphs_dir / 'loss.png', bbox_inches='tight', dpi=100)
    plt.close()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['train_dice'], 'b-', label='Train Dice', linewidth=2)
    ax.plot(epochs, history['val_dice'], 'r-', label='Val Dice', linewidth=2)
    ax.set_xlabel('Época')
    ax.set_ylabel('Dice Coefficient')
    ax.set_title('Dice Coefficient durante Entrenamiento')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graphs_dir / 'dice.png', bbox_inches='tight', dpi=100)
    plt.close()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['lr'], 'b-', linewidth=2)
    ax.set_xlabel('Época')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate durante Entrenamiento')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(graphs_dir / 'lr.png', bbox_inches='tight', dpi=100)
    plt.close()

training_start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_dice = validate_epoch(model, val_loader, criterion, device)
    
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)
    history['lr'].append(current_lr)
    
    save_training_graphs(history, RESULTS_DIR / 'graphs')
    
    scheduler.step()
    
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_model_path = RESULTS_DIR / 'bestModel' / 'best_model.pth'
        save_checkpoint(model, optimizer, epoch, train_loss, val_loss, train_dice, val_dice, best_model_path)
    
    if USE_WANDB:
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_dice": train_dice,
            "val_dice": val_dice,
            "learning_rate": current_lr,
            "best_val_dice": best_val_dice
        })
    
    elapsed = time.time() - training_start_time
    elapsed_str = f"{int(elapsed//3600)}h {int((elapsed%3600)//60)}m {int(elapsed%60)}s"
    
    clear_output(wait=True)
    print(f"Época {epoch}/{NUM_EPOCHS} | Train: L={train_loss:.4f} D={train_dice:.4f} | Val: L={val_loss:.4f} D={val_dice:.4f} | Best={best_val_dice:.4f} | LR={current_lr:.2e} | {elapsed_str}")

training_end_time = time.time()
training_duration_seconds = training_end_time - training_start_time

if USE_WANDB:
    wandb.finish()

print("Entrenamiento completado")

print(f"\nArchivos guardados:")
print(f"  - {RESULTS_DIR / 'graphs' / '*.png'}")
print(f"  - {RESULTS_DIR / 'bestModel' / 'best_model.pth'}")

history['training_duration_seconds'] = training_duration_seconds

hours = int(training_duration_seconds // 3600)
minutes = int((training_duration_seconds % 3600) // 60)
seconds = int(training_duration_seconds % 60)

print(f"Resumen del entrenamiento:")
print(f"  Duración total: {hours}h {minutes}m {seconds}s")
print(f"  Épocas completadas: {len(history['train_loss'])}")
print(f"  Mejor Train Dice: {max(history['train_dice']):.4f}")
print(f"  Mejor Val Dice: {max(history['val_dice']):.4f}")
print(f"  Train Loss final: {history['train_loss'][-1]:.4f}")
print(f"  Val Loss final: {history['val_loss'][-1]:.4f}")

### Visualizamos las curvas de entrenamiento

In [ ]:
try:
    _ = history['train_loss'][0]
    valores = history
    print("Usando valores desde memoria")
except (NameError, KeyError, IndexError):
    valores = None
    print("No hay datos de entrenamiento disponibles")

if valores is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    epochs = range(1, len(valores['train_loss']) + 1)

    ax1.plot(epochs, valores['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs, valores['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss durante Entrenamiento')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, valores['train_dice'], 'b-', label='Train Dice', linewidth=2)
    ax2.plot(epochs, valores['val_dice'], 'r-', label='Val Dice', linewidth=2)
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Dice Coefficient')
    ax2.set_title('Dice Coefficient durante Entrenamiento')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(valores['lr'], 'b-', linewidth=2)
    ax.set_xlabel('Época')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate durante Entrenamiento')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    plt.show()

    print(f"\nGráficos disponibles en: {RESULTS_DIR / 'graphs'}")
    print(f"  - loss.png")
    print(f"  - dice.png")
    print(f"  - lr.png")
else:
    print("Saltando gráficos - ejecutar entrenamiento primero")

## Evaluación del modelo
---

### Cargamos el mejor modelo guardado durante el entrenamiento

In [ ]:
best_model_path = RESULTS_DIR / 'bestModel' / 'best_model.pth'

checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Modelo cargado desde {best_model_path}")
print(f"  Época: {checkpoint['epoch']}")
print(f"  Train Loss: {checkpoint['train_loss']:.4f}")
print(f"  Val Loss: {checkpoint['val_loss']:.4f}")
print(f"  Train Dice: {checkpoint['train_dice']:.4f}")
print(f"  Val Dice: {checkpoint['val_dice']:.4f}")

model.eval()

### Optimización del threshold en el set de validación con TTA
- TTA = Test-Time Augmentation: predice imagen original + flipeada, promedia ambas
- Probamos diferentes thresholds para encontrar el que da mejor Dice

In [ ]:
USE_TTA = True

print(f"TTA (HorizontalFlip): {'ACTIVADO' if USE_TTA else 'DESACTIVADO'}")
print(f"Probando thresholds de {THRESHOLD_RANGE[0]:.2f} a {THRESHOLD_RANGE[-1]:.2f}")

threshold_results = {}

model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch_idx, (images, masks) in enumerate(val_loader):
        images = images.to(device)
        masks = masks.to(device)
        
        outputs = model(images)
        outputs = torch.sigmoid(outputs)
        
        if USE_TTA:
            images_flipped = torch.flip(images, dims=[3])
            outputs_flipped = model(images_flipped)
            outputs_flipped = torch.sigmoid(outputs_flipped)
            outputs_flipped = torch.flip(outputs_flipped, dims=[3])
            outputs = (outputs + outputs_flipped) / 2
        
        all_predictions.append(outputs.cpu())
        all_targets.append(masks.cpu())
        
        print(f"\r  Batch {batch_idx+1}/{len(val_loader)}", end="")

all_predictions = torch.cat(all_predictions, dim=0)
all_targets = torch.cat(all_targets, dim=0)

for threshold in THRESHOLD_RANGE:
    dice_scores = []
    
    for i in range(len(all_predictions)):
        pred = (all_predictions[i] > threshold).float()
        target = all_targets[i]
        
        dice = dice_coefficient(pred.unsqueeze(0), target.unsqueeze(0), threshold=0.5)
        dice_scores.append(dice)
    
    mean_dice = np.mean(dice_scores)
    threshold_results[threshold] = mean_dice
    
    print(f"  Threshold {threshold:.2f}: Dice = {mean_dice:.4f}")

optimal_threshold = max(threshold_results, key=threshold_results.get)
best_dice = threshold_results[optimal_threshold]
print(f"\nThreshold óptimo: {optimal_threshold:.2f} con Dice = {best_dice:.4f}")

### Evaluamos el modelo con el threshold óptimo. Calculamos Dice por cada imagen individual.

In [ ]:
print(f"Evaluando con threshold óptimo: {optimal_threshold:.2f}")

dice_per_image = []

for i in range(len(all_predictions)):
    pred = (all_predictions[i] > optimal_threshold).float()
    target = all_targets[i]
    
    dice = dice_coefficient(pred.unsqueeze(0), target.unsqueeze(0), threshold=0.5)
    dice_per_image.append(dice)

dice_per_image = np.array(dice_per_image)

print(f"\nEstadísticas de Dice en validación:")
print(f"  Media: {dice_per_image.mean():.4f}")
print(f"  Mediana: {np.median(dice_per_image):.4f}")
print(f"  Std: {dice_per_image.std():.4f}")
print(f"  Min: {dice_per_image.min():.4f}")
print(f"  Max: {dice_per_image.max():.4f}")
print(f"  Q25: {np.percentile(dice_per_image, 25):.4f}")
print(f"  Q75: {np.percentile(dice_per_image, 75):.4f}")

validation_results = {
    'optimal_threshold': float(optimal_threshold),
    'dice_mean': float(dice_per_image.mean()),
    'dice_median': float(np.median(dice_per_image)),
    'dice_std': float(dice_per_image.std()),
    'dice_min': float(dice_per_image.min()),
    'dice_max': float(dice_per_image.max()),
    'num_images': len(dice_per_image),
    'threshold_results': {float(k): float(v) for k, v in threshold_results.items()}
}

### Identificamos las mejores y peores predicciones y las visualizamos
- 5 mejores y 10 peores, se guardan en results/predictions/

In [ ]:
best_indices = np.argsort(dice_per_image)[-5:][::-1]
worst_indices = np.argsort(dice_per_image)[:10]

print("Mejores predicciones (Top 5):")
for i, idx in enumerate(best_indices):
    print(f"  {i+1}. Índice {idx}: Dice = {dice_per_image[idx]:.4f}")

print("\nPeores predicciones (Bottom 10):")
for i, idx in enumerate(worst_indices):
    print(f"  {i+1}. Índice {idx}: Dice = {dice_per_image[idx]:.4f}")

validation_results['best_indices'] = best_indices.tolist()
validation_results['worst_indices'] = worst_indices.tolist()
validation_results['best_dice_scores'] = [float(dice_per_image[i]) for i in best_indices]
validation_results['worst_dice_scores'] = [float(dice_per_image[i]) for i in worst_indices]

predictions_dir = RESULTS_DIR / 'predictions'

def visualize_prediction(image, pred_mask, true_mask, dice_score, title, save_path):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Imagen')
    axes[0].axis('off')
    
    axes[1].imshow(true_mask, cmap='gray')
    axes[1].set_title('Máscara Real')
    axes[1].axis('off')
    
    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title('Predicción')
    axes[2].axis('off')
    
    axes[3].imshow(image)
    axes[3].imshow(pred_mask, alpha=0.5, cmap='Reds')
    axes[3].set_title('Overlay')
    axes[3].axis('off')

    fig.suptitle(f'{title} (Dice: {dice_score:.4f})', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=100)
    plt.show()

print("\nVisualizando mejores predicciones (5)...")
for i, idx in enumerate(best_indices):
    img_path = val_img_paths[idx]
    mask_path = val_msk_paths[idx]
    
    image, true_mask = load_image_and_mask(img_path, mask_path)
    
    pred_mask = (all_predictions[idx, 0] > optimal_threshold).numpy()
    pred_mask_resized = cv2.resize(pred_mask.astype(np.float32), (ORIG_SIZE, ORIG_SIZE), interpolation=cv2.INTER_NEAREST)
    
    save_path = predictions_dir / f'best_{i+1:02d}.png'
    visualize_prediction(image, pred_mask_resized, true_mask, dice_per_image[idx], f'Mejor #{i+1}', save_path)

print("Visualizando peores predicciones (10)...")
for i, idx in enumerate(worst_indices):
    img_path = val_img_paths[idx]
    mask_path = val_msk_paths[idx]
    
    image, true_mask = load_image_and_mask(img_path, mask_path)
    
    pred_mask = (all_predictions[idx, 0] > optimal_threshold).numpy()
    pred_mask_resized = cv2.resize(pred_mask.astype(np.float32), (ORIG_SIZE, ORIG_SIZE), interpolation=cv2.INTER_NEAREST)
    
    save_path = predictions_dir / f'worst_{i+1:02d}.png'
    visualize_prediction(image, pred_mask_resized, true_mask, dice_per_image[idx], f'Peor #{i+1}', save_path)

print(f"\nVisualizaciones guardadas en {predictions_dir}")

### Creamos histograma de la distribución de Dice scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(dice_per_image, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(dice_per_image.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {dice_per_image.mean():.4f}')
ax.axvline(np.median(dice_per_image), color='green', linestyle='--', linewidth=2, label=f'Mediana: {np.median(dice_per_image):.4f}')
ax.set_xlabel('Dice Coefficient')
ax.set_ylabel('Frecuencia')
ax.set_title(f'Distribución de Dice Scores en Validación (threshold={optimal_threshold:.2f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
thresholds_list = sorted(threshold_results.keys())
dice_values = [threshold_results[t] for t in thresholds_list]
ax.plot(thresholds_list, dice_values, 'b-o', linewidth=2, markersize=8)
ax.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2, label=f'Óptimo: {optimal_threshold:.2f}')
ax.axhline(best_dice, color='green', linestyle='--', linewidth=2, alpha=0.5)
ax.set_xlabel('Threshold')
ax.set_ylabel('Dice Coefficient')
ax.set_title('Dice vs Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### Resumen de resultados de validación

In [ ]:
print(f"RESUMEN DE EVALUACIÓN")
print(f"Threshold óptimo: {optimal_threshold:.2f}")
print(f"Dice medio: {validation_results['dice_mean']:.4f}")
print(f"Dice mínimo: {validation_results['dice_min']:.4f}")
print(f"Dice máximo: {validation_results['dice_max']:.4f}")
print(f"Imágenes evaluadas: {validation_results['num_images']}")

### Creamos el dataset de test y el DataLoader

In [ ]:
test_dataset = SegmentationDataset(
    image_paths=test_image_paths,
    mask_paths=None,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Dataset de test creado:")
print(f"  Imágenes: {len(test_dataset)}")
print(f"  Batches: {len(test_loader)}")
print(f"  Batch size: {BATCH_SIZE}")

### Generamos predicciones para test CON TTA

In [ ]:
print(f"TTA (HorizontalFlip): {'ACTIVADO' if USE_TTA else 'DESACTIVADO'}")
print(f"Usando threshold óptimo: {optimal_threshold:.2f}")

model.eval()
test_predictions_256 = []

with torch.no_grad():
    for batch_idx, images in enumerate(test_loader):
        images = images.to(device)
        
        outputs = model(images)
        outputs = torch.sigmoid(outputs)
        
        if USE_TTA:
            images_flipped = torch.flip(images, dims=[3])
            outputs_flipped = model(images_flipped)
            outputs_flipped = torch.sigmoid(outputs_flipped)
            outputs_flipped = torch.flip(outputs_flipped, dims=[3])
            outputs = (outputs + outputs_flipped) / 2
        
        test_predictions_256.append(outputs.cpu())
        
        print(f"\r  Batch {batch_idx+1}/{len(test_loader)}", end="")

test_predictions_256 = torch.cat(test_predictions_256, dim=0)

print(f"\n\nPredicciones generadas:")
print(f"  Shape: {test_predictions_256.shape}")
print(f"  Total imágenes: {len(test_predictions_256)}")
print(f"  TTA: {'Sí (promedio de original + flip)' if USE_TTA else 'No'}")

### Aplicamos threshold a 256×256 y luego redimensionamos a 800×800

In [ ]:
test_predictions_binary = []

for i in range(len(test_predictions_256)):
    pred_256 = test_predictions_256[i, 0].numpy()
    
    binary_256 = (pred_256 > optimal_threshold).astype(np.uint8)
    
    binary_800 = cv2.resize(binary_256, (ORIG_SIZE, ORIG_SIZE), interpolation=cv2.INTER_NEAREST)
    
    test_predictions_binary.append(binary_800)
    
    if (i + 1) % 100 == 0:
        print(f"\r  Procesadas {i+1}/{len(test_predictions_256)}", end="")

test_predictions_binary = np.array(test_predictions_binary)

print(f"\n\nMáscaras binarias generadas:")
print(f"  Shape: {test_predictions_binary.shape}")
print(f"  Valores únicos: {np.unique(test_predictions_binary)}")
print(f"  Tipo: {test_predictions_binary.dtype}")

num_with_foreground = np.sum([pred.sum() > 0 for pred in test_predictions_binary])
print(f"\nImágenes con foreground detectado: {num_with_foreground}/{len(test_predictions_binary)}")

### Verificación de las máscaras binarias generadas

In [ ]:
print("Verificación de máscaras binarias:")
print(f"  Shape: {test_predictions_binary.shape}")
print(f"  Valores únicos: {np.unique(test_predictions_binary)}")
print(f"  Memoria: {test_predictions_binary.nbytes / (1024**2):.1f} MB")

foreground_ratios = [pred.sum() / pred.size for pred in test_predictions_binary]
print(f"\nEstadísticas de foreground en predicciones:")
print(f"  Media: {np.mean(foreground_ratios):.4f}")
print(f"  Min: {np.min(foreground_ratios):.4f}")
print(f"  Max: {np.max(foreground_ratios):.4f}")


### Visualizamos algunas predicciones de test para verificar

In [ ]:
num_samples = min(6, len(test_image_paths))
sample_indices = np.random.choice(len(test_image_paths), num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 5))

if num_samples == 1:
    axes = axes.reshape(1, -1)

for i, idx in enumerate(sample_indices):
    img_path = test_image_paths[idx]
    image = np.array(Image.open(img_path).convert('RGB'))
    pred_mask = test_predictions_binary[idx]
    
    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f'Test Imagen {test_ids[idx]}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(pred_mask, cmap='gray')
    axes[i, 1].set_title('Predicción')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(image)
    axes[i, 2].imshow(pred_mask, alpha=0.5, cmap='Reds')
    axes[i, 2].set_title('Overlay')
    axes[i, 2].axis('off')

plt.suptitle('Predicciones en Test Set', fontsize=16)
plt.tight_layout()
plt.show()

### RLE (Run-Length Encoding) para Kaggle
- Comprime máscara binaria en string: "start1 length1 start2 length2 ..."
- Kaggle espera columnas primero, Python usa filas por defecto (order='C')
- Sin order='F': predicciones aparecen rotadas 90° en el leaderboard

In [ ]:
def mask2rle(img):
    if not np.any(img):
        return ''
    
    pixels = img.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    
    return ' '.join(str(x) for x in runs)

def rle2mask(mask_rle, shape):
    if mask_rle == '':
        return np.zeros(shape, dtype=np.uint8)
    
    s = mask_rle.split()
    starts = np.asarray(s[0::2], dtype=int)
    lengths = np.asarray(s[1::2], dtype=int)
    starts -= 1
    ends = starts + lengths
    
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    
    return img.reshape(shape, order='F')

### Visualizamos un ejemplo de RLE encoding

In [ ]:
sample_idx = np.random.randint(len(test_predictions_binary))
sample_mask = test_predictions_binary[sample_idx]
sample_rle = mask2rle(sample_mask)
sample_reconstructed = rle2mask(sample_rle, sample_mask.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(sample_mask, cmap='gray')
axes[0].set_title('Máscara Original')
axes[0].axis('off')

axes[1].imshow(sample_reconstructed, cmap='gray')
axes[1].set_title('Reconstruida desde RLE')
axes[1].axis('off')

difference = np.abs(sample_mask.astype(int) - sample_reconstructed.astype(int))
axes[2].imshow(difference, cmap='Reds')
axes[2].set_title(f'Diferencia (sum={difference.sum()})')
axes[2].axis('off')

plt.suptitle(f'Test RLE - Imagen {test_ids[sample_idx]}', fontsize=16)
plt.tight_layout()
plt.show()

print(f"\nRLE string (primeros 200 chars):")
print(f"  {sample_rle[:200]}...")
print(f"  Longitud total: {len(sample_rle)} caracteres")

### Generamos RLE para todas las predicciones de test

In [ ]:
rle_encodings = []

for i in range(len(test_predictions_binary)):
    mask = test_predictions_binary[i]
    rle = mask2rle(mask)
    rle_encodings.append(rle)
    
    if (i + 1) % 100 == 0:
        print(f"\r  Procesadas {i+1}/{len(test_predictions_binary)}", end="")

print(f"\n\nRLE encodings generados:")
print(f"  Total: {len(rle_encodings)}")

empty_predictions = sum([1 for rle in rle_encodings if rle == ''])
print(f"  Predicciones vacías: {empty_predictions}")
print(f"  Predicciones con foreground: {len(rle_encodings) - empty_predictions}")

avg_rle_length = np.mean([len(rle) for rle in rle_encodings if rle != ''])
print(f"  Longitud promedio RLE: {avg_rle_length:.0f} caracteres")

### Creamos el DataFrame para submission de Kaggle
- Formato: id (nombre de archivo con extensión), encoded_pixels

In [ ]:
submission_data = {
    'id': [f"{id}.png" for id in test_ids],
    'encoded_pixels': rle_encodings
}

submission_df = pd.DataFrame(submission_data)

print(f"\nDataFrame creado:")
print(f"  Shape: {submission_df.shape}")
print(f"  Columnas: {list(submission_df.columns)}")

print(f"\nPrimeras filas:")
print(submission_df.head())

print(f"\nÚltimas filas:")
print(submission_df.tail())

submission_path = RESULTS_DIR / 'kaggleSubmission' / 'submission.csv'

submission_df.to_csv(submission_path, index=False)

print(f"Submission guardado en {submission_path}")

file_size_mb = submission_path.stat().st_size / (1024 * 1024)

print(f"  Tamaño: {file_size_mb:.2f} MB")
print(f"Archivo: {submission_path}")
print(f"Imágenes: {len(submission_df)}")
print(f"Threshold usado: {optimal_threshold:.2f}")
print(f"Val Dice obtenido: {validation_results['dice_mean']:.4f}")

## Métricas finales del modelo
---

In [ ]:
print("RESUMEN FINAL DEL MODELO")

print("\nARQUITECTURA:")
print(f"  Modelo: U-Net (4 niveles encoder/decoder)")
print(f"  Canales: 64 -> 128 -> 256 -> 512 -> 1024")
print(f"  Parámetros totales: {model_info['total_params']:,}")
print(f"  Dropout: 0.1 (en bottleneck)")
print(f"  Input/Output: {IMG_SIZE}x{IMG_SIZE}")

print("\nCONFIGURACION:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epocas maximas: {NUM_EPOCHS}")
print(f"  Val split: {VAL_SPLIT*100:.0f}%")
print(f"  Loss: BCEDiceLoss (BCE=0.3, Dice=0.7)")
print(f"  Optimizer: Adam")
print(f"  Scheduler: CosineAnnealingLR")

print("\nENTRENAMIENTO:")
try:
    _ = history['train_loss'][0]
    valores = history
except (NameError, KeyError, IndexError):
    valores = None

if valores is not None:
    print(f"  Epocas completadas: {len(valores['train_loss'])}")
    print(f"  Mejor Train Dice: {max(valores['train_dice']):.4f}")
    print(f"  Mejor Val Dice: {max(valores['val_dice']):.4f}")
    print(f"  Train Loss final: {valores['train_loss'][-1]:.4f}")
    print(f"  Val Loss final: {valores['val_loss'][-1]:.4f}")
    if 'training_duration_seconds' in valores:
        hours = int(valores['training_duration_seconds'] // 3600)
        minutes = int((valores['training_duration_seconds'] % 3600) // 60)
        print(f"  Duracion total: {hours}h {minutes}m")
else:
    print("  (Entrenamiento no ejecutado en esta sesion)")
    try:
        print(f"  Mejor modelo (checkpoint):")
        print(f"    Epoca: {checkpoint['epoch']}")
        print(f"    Train Dice: {checkpoint['train_dice']:.4f}")
        print(f"    Val Dice: {checkpoint['val_dice']:.4f}")
    except (NameError, KeyError):
        print("  No hay datos de entrenamiento disponibles")

print("\nEVALUACION (Validacion):")
print(f"  Threshold optimo: {optimal_threshold:.2f}")
print(f"  Dice medio: {validation_results['dice_mean']:.4f}")
print(f"  Dice std: {validation_results['dice_std']:.4f}")
print(f"  Dice minimo: {validation_results['dice_min']:.4f}")
print(f"  Dice maximo: {validation_results['dice_max']:.4f}")
print(f"  Imagenes evaluadas: {validation_results['num_images']}")

print("\nPREDICCIONES TEST:")
print(f"  Total imagenes: {len(test_predictions_binary)}")
print(f"  Con foreground: {len(rle_encodings) - empty_predictions}")
print(f"  Vacias (sin persona): {empty_predictions}")
print(f"  Tamano mascaras: {test_predictions_binary.shape[1]}x{test_predictions_binary.shape[2]}")

print("\nSUBMISSION KAGGLE:")
print(f"  Archivo: {submission_path}")
print(f"  Filas: {len(submission_df)}")
file_size_mb = submission_path.stat().st_size / (1024 * 1024)
print(f"  Tamano: {file_size_mb:.2f} MB")